In [1]:
# Install Java
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless

# Download Spark
!wget https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz

# Extract Spark
!tar -xvf spark-3.5.1-bin-hadoop3.tgz

# Install PySpark
!pip install pyspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jdk-headless is already the newest version (11.0.31+11-1ubuntu1~22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 44 not upgraded.
--2026-06-22 04:58:06--  https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400446614 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.1-bin-hadoop3.tgz.7’

spark-3.5.1-bin-had 100%[===================>] 381.90M   518KB/s    in 13m 27s 

2026-06-22 05:11:34 (485 KB/s) - ‘spark-3.5.1-bin-hadoop3.tgz.7’ sa

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Assignment") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.2


In [3]:
from google.colab import files

uploaded = files.upload()

Saving RetailSales_PySpark_Dataset.csv to RetailSales_PySpark_Dataset.csv


In [5]:
# Read the CSV dataset
df = spark.read.csv(
    "/content/RetailSales_PySpark_Dataset.csv",
    header=True,
    inferSchema=True
)

# Print the schema
df.printSchema()

# Display the first 5 rows
df.show(5)

root
 |-- CustomerID: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: integer (nullable = true)
 |-- Sales: integer (nullable = true)
 |-- OrderDate: date (nullable = true)

+----------+------------+----+------+------+-----------+----------+--------+---------+-----+----------+
|CustomerID|CustomerName| Age|Gender|Region|   Category|   Product|Quantity|UnitPrice|Sales| OrderDate|
+----------+------------+----+------+------+-----------+----------+--------+---------+-----+----------+
|      C001|       Aarav|25.0|  Male| North|Electronics|Headphones|       2|     1500| 3000|2026-01-05|
|      C002|        Diya|31.0|Female|  West|  Furniture|     Chair|       1|     4500| 4500|2026-01-06|
|      C003|       Kabir|22.0|  Male| 

In [6]:
from pyspark.sql.functions import col, lit, concat, lower

# Create required columns for the assignment
df = (df
      .withColumnRenamed("CustomerID", "user_id")
      .withColumnRenamed("OrderDate", "transaction_date")
      .withColumnRenamed("Category", "product_category")
      .withColumnRenamed("Sales", "sale_amount")
      .withColumn("price", col("sale_amount"))
      .withColumn("store_id", lit("Store_1"))
      .withColumn("subscription", lit("Premium"))
      .withColumn("status", lit(None).cast("string"))
      .withColumn("raw_timestamp", col("transaction_date").cast("string"))
      .withColumn("username", col("CustomerName"))
      .withColumn(
          "email",
          concat(lower(col("CustomerName")), lit("@gmail.com"))
      )
)

df.printSchema()
df.show(5)

root
 |-- user_id: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: integer (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- price: integer (nullable = true)
 |-- store_id: string (nullable = false)
 |-- subscription: string (nullable = false)
 |-- status: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)

+-------+------------+----+------+------+----------------+----------+--------+---------+-----------+----------------+-----+--------+------------+------+-------------+--------+----------------+
|user_id|CustomerName| Age|Gender|Region|product_category|   Pr

# Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Traditional MapReduce has several limitations that make Spark a better choice for modern big data processing.

1. It stores intermediate results on disk after every stage, which increases disk I/O and slows down processing.

2. It is not suitable for iterative tasks like Machine Learning because data is repeatedly read from and written to disk.

3. It has high latency, making it less efficient for real-time and interactive data analysis.

4. Developing complex workflows requires multiple MapReduce jobs, making the code more complex.

Apache Spark overcomes these limitations by using in-memory computing, DAG execution, faster processing, and easy-to-use APIs, making it a preferred framework for big data analytics.

# Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Apache Spark uses in-memory computing, which means it stores intermediate data in RAM instead of writing it to disk after every operation.

In iterative machine learning algorithms, the same dataset is processed multiple times. Since Spark keeps the data in memory, it avoids repeated disk read and write operations, resulting in much faster execution.

In contrast, traditional MapReduce stores intermediate results on disk after every iteration, increasing execution time due to high disk I/O. Therefore, Spark is more efficient for machine learning, graph processing, and interactive analytics.

# Q3. Remove duplicate rows based on user_id and transaction_date.

In [7]:
# Remove duplicate rows based on user_id and transaction_date
df_clean = df.dropDuplicates(["user_id", "transaction_date"])

# Display the cleaned DataFrame
df_clean.show(5)

+-------+------------+----+------+------+----------------+----------+--------+---------+-----------+----------------+-----+--------+------------+------+-------------+--------+----------------+
|user_id|CustomerName| Age|Gender|Region|product_category|   Product|Quantity|UnitPrice|sale_amount|transaction_date|price|store_id|subscription|status|raw_timestamp|username|           email|
+-------+------------+----+------+------+----------------+----------+--------+---------+-----------+----------------+-----+--------+------------+------+-------------+--------+----------------+
|   C001|       Aarav|25.0|  Male| North|     Electronics|Headphones|       2|     1500|       3000|      2026-01-05| 3000| Store_1|     Premium|  NULL|   2026-01-05|   Aarav| aarav@gmail.com|
|   C002|        Diya|31.0|Female|  West|       Furniture|     Chair|       1|     4500|       4500|      2026-01-06| 4500| Store_1|     Premium|  NULL|   2026-01-06|    Diya|  diya@gmail.com|
|   C003|       Kabir|22.0|  Male| 

# Q4. Filter the dataset where Region is 'West' and calculate the average sale_amount for each product_category.

In [8]:
from pyspark.sql.functions import avg

# Filter records where Region is 'West'
west_sales = df.filter(df.Region == "West")

# Group by product_category and calculate average sale_amount
result = west_sales.groupBy("product_category") \
                   .agg(avg("sale_amount").alias("average_sale_amount"))

# Display the result
result.show()

+----------------+-------------------+
|product_category|average_sale_amount|
+----------------+-------------------+
|     Electronics|             5600.0|
|       Furniture|             9900.0|
+----------------+-------------------+



# Q5. Difference between .na.drop() and .na.fill()

The `.na.drop()` function removes rows that contain null values from a DataFrame.

The `.na.fill()` function replaces null values with a specified value instead of removing the rows.

Using `.na.fill()` is useful when we want to keep all records while replacing missing values with meaningful defaults.

In this example, the null values in the `status` column are replaced with the string `"Unknown"`.

In [9]:
# Fill null values in the 'status' column with 'Unknown'
df_filled = df.na.fill({"status": "Unknown"})

# Display selected columns to verify the changes
df_filled.select("user_id", "status").show(10)

+-------+-------+
|user_id| status|
+-------+-------+
|   C001|Unknown|
|   C002|Unknown|
|   C003|Unknown|
|   C004|Unknown|
|   C005|Unknown|
|   C006|Unknown|
|   C007|Unknown|
|   C008|Unknown|
|   C009|Unknown|
|   C010|Unknown|
+-------+-------+
only showing top 10 rows


# Q6. Find the total count of records for each city where the count is greater than 1.

In [10]:
from pyspark.sql.functions import when, col

# Create a city column based on Region
df = df.withColumn(
    "city",
    when(col("Region") == "North", "Delhi")
    .when(col("Region") == "South", "Chennai")
    .when(col("Region") == "East", "Kolkata")
    .otherwise("Mumbai")
)

# Display the first few rows
df.select("Region", "city").show(5)

+------+-------+
|Region|   city|
+------+-------+
| North|  Delhi|
|  West| Mumbai|
| South|Chennai|
|  East|Kolkata|
| North|  Delhi|
+------+-------+
only showing top 5 rows


In [13]:
city_count = df.groupBy("city") \
               .agg(count("*").alias("total_records")) \
               .filter(col("total_records") > 1)

city_count.show()

+-------+-------------+
|   city|total_records|
+-------+-------------+
|Chennai|            4|
| Mumbai|            7|
|Kolkata|            4|
|  Delhi|            5|
+-------+-------------+



# Q7. How does the immutability of Spark DataFrames affect data cleaning?

Spark DataFrames are immutable, which means they cannot be modified after they are created.

When performing data cleaning operations such as dropping columns, renaming columns, or filtering rows, Spark does not change the original DataFrame. Instead, it creates and returns a new DataFrame with the required changes.

This approach ensures data consistency, supports fault tolerance, and allows Spark to optimize execution efficiently.

In [14]:
# Drop the 'Gender' column
df_new = df.drop("Gender")

# Rename 'UnitPrice' to 'Price'
df_new = df_new.withColumnRenamed("UnitPrice", "Price")

# Display the updated schema
df_new.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price: integer (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- price: integer (nullable = true)
 |-- store_id: string (nullable = false)
 |-- subscription: string (nullable = false)
 |-- status: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = false)



# Q8. Filter the dataset where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [15]:
# Filter customers whose age is between 18 and 30
# and whose subscription is 'Premium'

filtered_df = df.filter(
    (df.Age.between(18, 30)) &
    (df.subscription == "Premium")
)

filtered_df.show()

+-------+------------+----+------+------+----------------+----------+--------+---------+-----------+----------------+-----+--------+------------+------+-------------+--------+----------------+-------+
|user_id|CustomerName| Age|Gender|Region|product_category|   Product|Quantity|UnitPrice|sale_amount|transaction_date|price|store_id|subscription|status|raw_timestamp|username|           email|   city|
+-------+------------+----+------+------+----------------+----------+--------+---------+-----------+----------------+-----+--------+------------+------+-------------+--------+----------------+-------+
|   C001|       Aarav|25.0|  Male| North|     Electronics|Headphones|       2|     1500|       3000|      2026-01-05| 3000| Store_1|     Premium|  NULL|   2026-01-05|   Aarav| aarav@gmail.com|  Delhi|
|   C003|       Kabir|22.0|  Male| South|         Grocery|      Rice|       5|       80|        400|      2026-01-06|  400| Store_1|     Premium|  NULL|   2026-01-06|   Kabir| kabir@gmail.com|Chen

# Q9. Why is it better to handle null values before performing mathematical aggregations?

It is important to handle null values before performing mathematical aggregations such as `sum()` or `avg()` because null values can lead to incorrect or incomplete results.

If null values are not handled properly:
- The calculated average may not accurately represent the data.
- Missing values can affect the quality of analysis.
- Reports and business decisions based on the data may become unreliable.

By replacing or removing null values before aggregation, we can ensure more accurate and meaningful results.

In [16]:
from pyspark.sql.functions import avg

# Fill null values in the 'price' column with 0
df_null_handled = df.na.fill({"price": 0})

# Calculate the average price
df_null_handled.select(avg("price").alias("Average_Price")).show()

+-------------+
|Average_Price|
+-------------+
|       7642.5|
+-------------+



# Q10. Cast the `raw_timestamp` column to `TimestampType` and rename it to `event_time`.

In [17]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

# Create a new column 'event_time' by casting raw_timestamp to TimestampType
df_timestamp = df.withColumn(
    "event_time",
    col("raw_timestamp").cast(TimestampType())
)

# Remove the old raw_timestamp column
df_timestamp = df_timestamp.drop("raw_timestamp")

# Display the schema
df_timestamp.printSchema()

# Display the first 5 rows
df_timestamp.select("user_id", "event_time").show(5)

root
 |-- user_id: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: integer (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- price: integer (nullable = true)
 |-- store_id: string (nullable = false)
 |-- subscription: string (nullable = false)
 |-- status: string (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = false)
 |-- event_time: timestamp (nullable = true)

+-------+-------------------+
|user_id|         event_time|
+-------+-------------------+
|   C001|2026-01-05 00:00:00|
|   C002|2026-01-06 00:00:00|
|   C003|2026-01-06 00:00:00|
|   C004|2026-01-07 00:00:00|
|   C005|

# Q11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

Shuffle is the process of redistributing data across different partitions during operations such as `groupBy()`, `join()`, `distinct()`, and `orderBy()`.

During a grouping operation, Spark moves records with the same key to the same partition so that aggregation can be performed correctly.

Shuffle is considered a wide transformation because data is exchanged between multiple partitions across different executors. This involves network communication and disk I/O, making it more expensive than narrow transformations.

Although shuffle increases processing time, it is necessary for operations that require data from multiple partitions.

In [18]:
from pyspark.sql.functions import sum

# Group data by product category
shuffle_df = df.groupBy("product_category") \
               .agg(sum("sale_amount").alias("Total_Sales"))

shuffle_df.show()

+----------------+-----------+
|product_category|Total_Sales|
+----------------+-----------+
|         Fashion|      11100|
|         Grocery|       3650|
|     Electronics|      88600|
|       Furniture|      49500|
+----------------+-----------+



# Q12. Identify and remove rows where the email column contains null values or the username is an empty string.

In [19]:
from pyspark.sql.functions import when, col

# Create sample null email and empty username values
df_test = df.withColumn(
    "email",
    when(col("user_id") == "C003", None).otherwise(col("email"))
).withColumn(
    "username",
    when(col("user_id") == "C004", "").otherwise(col("username"))
)

# Display the modified records
df_test.select("user_id", "email", "username").show()

+-------+----------------+--------+
|user_id|           email|username|
+-------+----------------+--------+
|   C001| aarav@gmail.com|   Aarav|
|   C002|  diya@gmail.com|    Diya|
|   C003|            NULL|   Kabir|
|   C004| meera@gmail.com|        |
|   C005|ishaan@gmail.com|  Ishaan|
|   C006| anaya@gmail.com|   Anaya|
|   C007| rohan@gmail.com|   Rohan|
|   C008|  sara@gmail.com|    Sara|
|   C009|vivaan@gmail.com|  Vivaan|
|   C010| kiara@gmail.com|   Kiara|
|   C011| arjun@gmail.com|   Arjun|
|   C012| nisha@gmail.com|   Nisha|
|   C013|   dev@gmail.com|     Dev|
|   C014|  riya@gmail.com|    Riya|
|   C015|  yash@gmail.com|    Yash|
|   C016| pooja@gmail.com|   Pooja|
|   C017| karan@gmail.com|   Karan|
|   C018|  neha@gmail.com|    Neha|
|   C019|  aman@gmail.com|    Aman|
|   C020|  diya@gmail.com|    Diya|
+-------+----------------+--------+



In [20]:
# Remove rows where email is null OR username is empty
clean_df = df_test.filter(
    (col("email").isNotNull()) &
    (col("username") != "")
)

# Display the cleaned DataFrame
clean_df.select("user_id", "email", "username").show()

+-------+----------------+--------+
|user_id|           email|username|
+-------+----------------+--------+
|   C001| aarav@gmail.com|   Aarav|
|   C002|  diya@gmail.com|    Diya|
|   C005|ishaan@gmail.com|  Ishaan|
|   C006| anaya@gmail.com|   Anaya|
|   C007| rohan@gmail.com|   Rohan|
|   C008|  sara@gmail.com|    Sara|
|   C009|vivaan@gmail.com|  Vivaan|
|   C010| kiara@gmail.com|   Kiara|
|   C011| arjun@gmail.com|   Arjun|
|   C012| nisha@gmail.com|   Nisha|
|   C013|   dev@gmail.com|     Dev|
|   C014|  riya@gmail.com|    Riya|
|   C015|  yash@gmail.com|    Yash|
|   C016| pooja@gmail.com|   Pooja|
|   C017| karan@gmail.com|   Karan|
|   C018|  neha@gmail.com|    Neha|
|   C019|  aman@gmail.com|    Aman|
|   C020|  diya@gmail.com|    Diya|
+-------+----------------+--------+



# Q13. Use the .agg() function to calculate the minimum, maximum, and average of the price column.

The `.agg()` function is used to perform multiple aggregate operations on one or more columns in a single statement. It helps calculate statistics such as minimum, maximum, average, sum, and count efficiently.

In [21]:
from pyspark.sql.functions import min, max, avg

# Calculate minimum, maximum, and average price
result = df.agg(
    min("price").alias("Minimum_Price"),
    max("price").alias("Maximum_Price"),
    avg("price").alias("Average_Price")
)

# Display the result
result.show()

+-------------+-------------+-------------+
|Minimum_Price|Maximum_Price|Average_Price|
+-------------+-------------+-------------+
|          330|        55000|       7642.5|
+-------------+-------------+-------------+



# Q14. What is the risk of using inferSchema=true when the dataset contains inconsistent date formats?

The `inferSchema=true` option automatically detects the data type of each column. However, if a dataset contains inconsistent or messy date formats, Spark may infer the wrong data type.

For example, if some dates are stored as `2026-01-05` and others as `05/01/2026`, Spark may treat the entire column as a string instead of a date.

This can lead to:
- Incorrect data type detection.
- Errors while filtering or sorting dates.
- Incorrect results in date calculations.
- Additional data cleaning effort.

Therefore, for messy datasets, it is better to define the schema explicitly instead of relying on `inferSchema=true`.

# Q15. Final processing pipeline

This pipeline performs the following operations:

1. Removes duplicate records.
2. Replaces null values in the `price` column with 0.
3. Groups the data by `store_id`.
4. Calculates the total revenue for each store.

In [22]:
from pyspark.sql.functions import sum

# Final processing pipeline
final_result = (
    df.dropDuplicates()
      .na.fill({"price": 0})
      .groupBy("store_id")
      .agg(sum("price").alias("total_revenue"))
)

# Display the result
final_result.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
| Store_1|       152850|
+--------+-------------+

